# Hybrid-Feature Neural Network (Constance Chen)

This experiment combines word TF-IDF, character TF-IDF, and
handcrafted writing-style features in a PyTorch neural network.

The goal is to investigate whether learning interactions between
these representations improves AI-vs-human text classification.

In [74]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.sparse import hstack, csr_matrix

import torch
from torch import nn

import time

import copy

from sklearn.base import clone


In [72]:
# locate project folder
project_dir = Path.cwd().resolve()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir))

from data_utils import load_data, split_data

# use same dataset as Q1, Q2, and Q3
zip_path = project_dir / "data" / "cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip"

train, unlabeled, test = load_data(zip_path)

# preserve same stratified validation split
train_df, val_df = split_data(train)

print(f"Training examples: {len(train_df)}")
print(f"Validation examples: {len(val_df)}")
print(f"Kaggle test examples: {len(test)}")

Training examples: 8000
Validation examples: 2000
Kaggle test examples: 3000


## Feature Extraction

We combine word-level TF-IDF, character-level TF-IDF, and
handcrafted writing-style features.

Word TF-IDF captures vocabulary and phrases, while character
TF-IDF captures spelling, punctuation, and stylistic patterns.

All vectorizers are fitted only on the training partition to
prevent validation leakage.

In [73]:

# Word TF-IDF: captures individual words and short phrases
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=3,
    max_features=20000,
    sublinear_tf=True
)

# Character TF-IDF: captures character patterns within word boundaries
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    lowercase=False,
    ngram_range=(3, 5),
    min_df=5,
    max_features=30000,
    sublinear_tf=True
)

# Fit only on training data, then transform validation data
X_word_train = word_vectorizer.fit_transform(train_df["text"])
X_word_val = word_vectorizer.transform(val_df["text"])

X_char_train = char_vectorizer.fit_transform(train_df["text"])
X_char_val = char_vectorizer.transform(val_df["text"])

print("Word TF-IDF:")
print("Train:", X_word_train.shape)
print("Validation:", X_word_val.shape)

print("\nCharacter TF-IDF:")
print("Train:", X_char_train.shape)
print("Validation:", X_char_val.shape)


KeyboardInterrupt: 

In [ ]:
from style_feature_utils import features as extract_style_features

# extract mia's 99 style features from raw text
X_style_train_raw = np.array(
    [extract_style_features(t) for t in train_df["text"]],
    dtype=np.float32
)

X_style_val_raw = np.array(
    [extract_style_features(t) for t in val_df["text"]],
    dtype=np.float32
)

# fit scaler on training features only
style_scaler = StandardScaler()

X_style_train = csr_matrix(
    style_scaler.fit_transform(X_style_train_raw)
)

X_style_val = csr_matrix(
    style_scaler.transform(X_style_val_raw)
)

# verify feature dimensions and values
assert X_style_train.shape[1] == 99
assert np.isfinite(X_style_train.data).all()
assert np.isfinite(X_style_val.data).all()

print("Style features:")
print("Train:", X_style_train.shape)
print("Validation:", X_style_val.shape)


Style features:
Train: (8000, 99)
Validation: (2000, 99)


### Combine Feature Representations

We concatenate word TF-IDF, character TF-IDF, and standardized
writing-style features into a single feature matrix.

The resulting matrix will be used as input to our neural network.

In [ ]:

# combine all three representations horizontally
X_train = hstack([
    X_word_train,
    X_char_train,
    X_style_train
], format="csr")

X_val = hstack([
    X_word_val,
    X_char_val,
    X_style_val
], format="csr")

# labels: 0 = human, 1 = AI
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()

# verify shapes
print("Combined training shape:", X_train.shape)
print("Combined validation shape:", X_val.shape)

assert X_train.shape[1] == X_val.shape[1]
assert len(y_train) == X_train.shape[0]
assert len(y_val) == X_val.shape[0]


Combined training shape: (8000, 50099)
Combined validation shape: (2000, 50099)


## Neural Network Implementation

We use a PyTorch multilayer perceptron (MLP) to classify
AI-generated and human-written text.

To reduce memory usage, sparse feature matrices are converted
into dense tensors one batch at a time.

In [ ]:

# prepare sparse matrices for PyTorch
class SparseTextDataset(torch.utils.data.Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # return row index; convert features in batches
        return idx

    def collate(self, indices):
        # convert only the current batch to a dense tensor
        xb = torch.tensor(
            self.X[indices].toarray(),
            dtype=torch.float32
        )

        if self.y is None:
            return xb

        yb = torch.tensor(
            self.y[indices],
            dtype=torch.float32
        )

        return xb, yb


# create data loaders
def make_loader(X, y=None, batch_size=16, shuffle=False):
    dataset = SparseTextDataset(X, y)

    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=dataset.collate
    )


# shuffle training examples, but not validation examples
train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)

# check a single training batch
xb, yb = next(iter(train_loader))

print("Feature batch:", xb.shape)
print("Label batch:", yb.shape)


Feature batch: torch.Size([16, 50099])
Label batch: torch.Size([16])


### Neural Network Architecture

We use a multilayer perceptron (MLP) with two hidden layers
containing 256 and 64 neurons. ReLU activations introduce
nonlinearity, while dropout (0.5) helps reduce overfitting.

The output layer produces one logit per passage for binary
classification.

In [ ]:
# fixed random seed for reproducibility
torch.manual_seed(42)

class HybridMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        # return one raw prediction score per passage
        return self.network(x).squeeze(-1)


# create model using the combined feature dimension
model = HybridMLP(input_dim=X_train.shape[1])

print(model)

# test forward pass without training
model.eval()
with torch.no_grad():
    logits = model(xb)

print("\nOutput shape:", logits.shape)
print("Total parameters:", sum(p.numel() for p in model.parameters()))


HybridMLP(
  (network): Sequential(
    (0): Linear(in_features=50099, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

Output shape: torch.Size([16])
Total parameters: 12842113


## Model Training

We use binary cross-entropy loss with logits for binary classification
and the AdamW optimizer with weight decay for regularization.

We first test a single training step before training the full model.

In [ ]:

# binary classification loss
loss_fn = nn.BCEWithLogitsLoss()

# optimizer with weight decay to reduce overfitting
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# use smaller batches to limit memory usage
train_loader = make_loader(
    X_train, y_train,
    batch_size=16,
    shuffle=True
)

# get one training batch
xb, yb = next(iter(train_loader))

# enable training mode (activates dropout)
model.train()

# reset previous gradients
optimizer.zero_grad()

# forward pass: predict raw logits
logits = model(xb)

# calculate loss against true labels
loss = loss_fn(logits, yb)

# backward pass: calculate gradients
loss.backward()

# update model weights
optimizer.step()

print(f"One training step complete. Loss: {loss.item():.4f}")


One training step complete. Loss: 0.6923


### Training Speed Test

We test a small number of batches to estimate the computational
cost of training the hybrid neural network.

In [ ]:
# test training speed on 10 batches
model.train()
start = time.perf_counter()

num_batches = 10
losses = []

for i, (xb, yb) in enumerate(train_loader):
    if i >= num_batches:
        break

    optimizer.zero_grad()

    logits = model(xb)
    loss = loss_fn(logits, yb)

    loss.backward()
    optimizer.step()

    losses.append(loss.item())

elapsed = time.perf_counter() - start

print(f"Completed {len(losses)} batches")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Average time per batch: {elapsed / len(losses):.2f} seconds")
print(f"Average loss: {np.mean(losses):.4f}")

# estimate full epoch duration (8,000 training examples)
batches_per_epoch = len(train_loader)
estimated_minutes = (
    elapsed / len(losses) * batches_per_epoch / 60
)

print(f"Estimated time per epoch: {estimated_minutes:.1f} minutes")


Completed 10 batches
Total time: 0.42 seconds
Average time per batch: 0.04 seconds
Average loss: 0.7026
Estimated time per epoch: 0.4 minutes


### Training and Validation

We train the hybrid MLP using binary cross-entropy loss and
the AdamW optimizer. After each epoch, we evaluate the model
on the fixed held-out validation set using AUROC.

We retain the weights from the epoch with the lowest
validation loss to reduce overfitting.

In [ ]:
# start with a fresh model, since the speed test updated the previous one
torch.manual_seed(42)
model = HybridMLP(input_dim=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# use small batches for memory efficiency
train_loader = make_loader(X_train, y_train, batch_size=16, shuffle=True)
val_loader = make_loader(X_val, y_val, batch_size=16, shuffle=False)

epochs = 5
history = []
best_val_loss = float("inf")
best_state = None
best_epoch = None

for epoch in range(1, epochs + 1):
    start = time.perf_counter()

    # training
    model.train()
    total_train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        logits = model(xb)
        loss = loss_fn(logits, yb)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(yb)

    train_loss = total_train_loss / len(y_train)

    # validation: no weight updates
    model.eval()
    total_val_loss = 0.0
    val_probs = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_val_loss += loss.item() * len(yb)
            val_probs.append(torch.sigmoid(logits).numpy())

    val_loss = total_val_loss / len(y_val)
    val_probs = np.concatenate(val_probs)
    val_auc = roc_auc_score(y_val, val_probs)

    # preserve the weights with the lowest validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch

    elapsed = time.perf_counter() - start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc
    })

    print(
        f"Epoch {epoch}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Val Loss = {val_loss:.4f}, "
        f"Val AUROC = {val_auc:.4f}, "
        f"Time = {elapsed:.1f}s"
    )

# restore the best epoch
model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")


Epoch 1: Train Loss = 0.5181, Val Loss = 0.2920, Val AUROC = 0.9590, Time = 20.1s
Epoch 2: Train Loss = 0.2050, Val Loss = 0.1660, Val AUROC = 0.9840, Time = 22.1s
Epoch 3: Train Loss = 0.1045, Val Loss = 0.1216, Val AUROC = 0.9912, Time = 19.8s
Epoch 4: Train Loss = 0.0570, Val Loss = 0.1087, Val AUROC = 0.9922, Time = 20.9s
Epoch 5: Train Loss = 0.0328, Val Loss = 0.0935, Val AUROC = 0.9941, Time = 21.8s

Best epoch: 5
Best validation loss: 0.0935


In [ ]:
# continue training for 5 more epochs
# keep the existing model and optimizer

for epoch in range(6, 11):
    start = time.perf_counter()

    model.train()
    total_train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        logits = model(xb)
        loss = loss_fn(logits, yb)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(yb)

    train_loss = total_train_loss / len(y_train)

    # evaluate without updating weights
    model.eval()
    total_val_loss = 0.0
    val_probs = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_val_loss += loss.item() * len(yb)
            val_probs.append(torch.sigmoid(logits).numpy())

    val_loss = total_val_loss / len(y_val)
    val_probs = np.concatenate(val_probs)
    val_auc = roc_auc_score(y_val, val_probs)

    # save the best weights
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc
    })

    elapsed = time.perf_counter() - start

    print(
        f"Epoch {epoch}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Val Loss = {val_loss:.4f}, "
        f"Val AUROC = {val_auc:.4f}, "
        f"Time = {elapsed:.1f}s"
    )

# restore best-performing weights
model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")


Epoch 6: Train Loss = 0.0191, Val Loss = 0.0946, Val AUROC = 0.9939, Time = 18.3s
Epoch 7: Train Loss = 0.0111, Val Loss = 0.1031, Val AUROC = 0.9934, Time = 18.1s
Epoch 8: Train Loss = 0.0073, Val Loss = 0.1001, Val AUROC = 0.9938, Time = 18.0s
Epoch 9: Train Loss = 0.0052, Val Loss = 0.1075, Val AUROC = 0.9939, Time = 20.9s
Epoch 10: Train Loss = 0.0041, Val Loss = 0.1083, Val AUROC = 0.9942, Time = 20.4s

Best epoch: 5
Best validation loss: 0.0935


### Final Validation Evaluation

We select the model from epoch 5, which achieved the lowest
validation loss. Training beyond this point resulted in
higher validation loss without meaningful AUROC improvement.

In [ ]:

# evaluate the selected model on the validation set
model.eval()
final_val_probs = []

with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb)
        probs = torch.sigmoid(logits)
        final_val_probs.append(probs.numpy())

final_val_probs = np.concatenate(final_val_probs)

# calculate AUROC for the restored model
final_val_auc = roc_auc_score(y_val, final_val_probs)

print(f"Selected epoch: {best_epoch}")
print(f"Final validation AUROC: {final_val_auc:.4f}")


Selected epoch: 5
Final validation AUROC: 0.9941


In [79]:
# save validation predictions for future model blending
hybrid_val_df = pd.DataFrame({
    "id": val_df["id"].to_numpy(),
    "label": val_df["label"].to_numpy(),
    "hybrid_prob": final_val_probs
})

# verify predictions correspond to the validation examples
assert len(hybrid_val_df) == 2000
assert hybrid_val_df["id"].is_unique
assert hybrid_val_df["hybrid_prob"].between(0, 1).all()

# save locally; do not commit prediction CSV files
output_path = project_dir / "hybrid_val_predictions.csv"
hybrid_val_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(hybrid_val_df.shape)
print(hybrid_val_df.head())


Saved to: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38/hybrid_val_predictions.csv
(2000, 3)
     id  label  hybrid_prob
0  5865      0     0.720152
1  1774      1     1.000000
2  3336      0     0.002063
3  9082      1     0.998574
4  6970      0     0.000395


## Kaggle Submission

We retrain the selected hybrid MLP configuration on all 10,000 labeled examples using five epochs.

We refit the word and character TF-IDF vectorizers and writing-style feature scaler on the full labeled dataset, then transform the 3,000 Kaggle test examples using the same fitted transformations.

In [75]:
# create fresh vectorizers with the same hyperparameters
full_word_vectorizer = clone(word_vectorizer)
full_char_vectorizer = clone(char_vectorizer)
full_style_scaler = StandardScaler()

# word TF-IDF
X_word_full = full_word_vectorizer.fit_transform(train["text"])
X_word_test = full_word_vectorizer.transform(test["text"])

# character TF-IDF
X_char_full = full_char_vectorizer.fit_transform(train["text"])
X_char_test = full_char_vectorizer.transform(test["text"])

# extract style features from the full dataset
X_style_full_raw = np.array(
    [extract_style_features(t) for t in train["text"]],
    dtype=np.float32
)

X_style_test_raw = np.array(
    [extract_style_features(t) for t in test["text"]],
    dtype=np.float32
)

# fit scaler only on labeled training data
X_style_full = csr_matrix(
    full_style_scaler.fit_transform(X_style_full_raw)
)

X_style_test = csr_matrix(
    full_style_scaler.transform(X_style_test_raw)
)

# combine all three feature representations
X_full = hstack([
    X_word_full,
    X_char_full,
    X_style_full
], format="csr")

X_test = hstack([
    X_word_test,
    X_char_test,
    X_style_test
], format="csr")

y_full = train["label"].to_numpy()

# verify shapes and values
assert X_full.shape[0] == 10000
assert X_test.shape[0] == 3000
assert X_full.shape[1] == X_test.shape[1]
assert np.isfinite(X_full.data).all()
assert np.isfinite(X_test.data).all()

print("Full training shape:", X_full.shape)
print("Kaggle test shape:", X_test.shape)
print("Training labels:", len(y_full))


Full training shape: (10000, 50099)
Kaggle test shape: (3000, 50099)
Training labels: 10000


### Final Model Training

We retrain a fresh hybrid MLP on all 10,000 labeled examples,
using the same architecture, learning rate, regularization,
and five training epochs selected during validation.

In [76]:
# use a fixed random seed for reproducibility
torch.manual_seed(42)

# create a fresh model for the full dataset
full_model = HybridMLP(input_dim=X_full.shape[1])

# binary classification loss
full_loss_fn = nn.BCEWithLogitsLoss()

# use the same optimizer settings as validation
full_optimizer = torch.optim.AdamW(
    full_model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# use all 10,000 labeled examples
full_loader = make_loader(
    X_full,
    y_full,
    batch_size=16,
    shuffle=True
)

# train for the selected number of epochs
final_epochs = best_epoch

for epoch in range(1, final_epochs + 1):
    start = time.perf_counter()

    full_model.train()
    total_loss = 0.0

    for xb, yb in full_loader:
        # reset gradients
        full_optimizer.zero_grad()

        # forward pass
        logits = full_model(xb)

        # calculate loss
        loss = full_loss_fn(logits, yb)

        # backpropagation and weight update
        loss.backward()
        full_optimizer.step()

        total_loss += loss.item() * len(yb)

    avg_loss = total_loss / len(y_full)
    elapsed = time.perf_counter() - start

    print(
        f"Epoch {epoch}/{final_epochs}: "
        f"Train Loss = {avg_loss:.4f}, "
        f"Time = {elapsed:.1f}s"
    )

print("\nFinal model training complete!")


Epoch 1/5: Train Loss = 0.4730, Time = 25.3s
Epoch 2/5: Train Loss = 0.1692, Time = 22.5s
Epoch 3/5: Train Loss = 0.0813, Time = 23.2s
Epoch 4/5: Train Loss = 0.0448, Time = 20.7s
Epoch 5/5: Train Loss = 0.0231, Time = 20.7s

Final model training complete!


### Generate Kaggle Predictions

We use the final model to predict AI-generated text probabilities
for the 3,000 unlabeled Kaggle test examples.

The predictions are saved in the required CSV format with
columns `id` and `label`.

In [77]:

# switch to evaluation mode (disables dropout)
full_model.eval()

# prepare Kaggle test data without shuffling
test_loader = make_loader(
    X_test,
    batch_size=16,
    shuffle=False
)

# predict probability of AI-generated text
test_probs = []

with torch.no_grad():
    for xb in test_loader:
        logits = full_model(xb)
        probs = torch.sigmoid(logits)
        test_probs.append(probs.cpu().numpy())

test_probs = np.concatenate(test_probs)

# match probabilities to original Kaggle test IDs
submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "label": test_probs
})

# verify submission format and predictions
assert len(submission) == 3000
assert submission["id"].is_unique
assert submission["label"].notna().all()
assert np.isfinite(submission["label"]).all()
assert submission["label"].between(0, 1).all()

# save locally (do not commit the CSV to GitHub)
output_path = project_dir / "submission_hybrid_mlp.csv"
submission.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Submission shape: {submission.shape}")
print(submission.head())

Saved to: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38/submission_hybrid_mlp.csv
Submission shape: (3000, 2)
      id     label
0  13001  1.000000
1  13002  0.001738
2  13003  0.082627
3  13004  0.001795
4  13005  0.999989


## Results and Discussion

We trained a hybrid PyTorch MLP using word TF-IDF (20,000 features), character TF-IDF (30,000 features), and 99 handcrafted writing-style features.

The network uses two hidden layers (256 and 64 neurons), ReLU activations, dropout of 0.5, and the AdamW optimizer.

We evaluated training through 10 epochs and selected epoch 5 based on the lowest validation loss.

| Metric | Result |
|---|---:|
| Selected epoch | 5 |
| Validation AUROC | 0.994118 |
| Public Kaggle AUROC | 0.96013 |

For the final Kaggle submission, we refitted the feature transformations and trained a fresh model for five epochs using all 10,000 labeled examples.

The hybrid MLP improved over our earlier standalone models and was subsequently incorporated into the ensemble experiments in Notebook 07.

The difference between validation and public Kaggle performance suggests that the random validation split may not fully represent the hidden test distribution.
